Самостоятельно реализуйте DecisionTree с некоторыми ограничениями (перечислены ниже). Задание может быть оценено частично.

Он должен получать на вход матрицу объект-признак и целевую переменную. На каждом этапе должен производить определение оптимальной пары признак-величина (для создания разделения данных, поступавших в вершину). Ограничение обучения реализуйте через ограничение допустимой максимальной глубины дерева - остальные опционарно.

**Рассмотрим игрушечную задачу бинарной классификации: поедет ли с Вами новый знакомый из бара? Это будет зависеть от Вашей внешности и красноречия, крепости предлагаемых напитков и, как это ни меркантильно, от количества потраченных в баре денег.**

### Создание набора данных

In [1]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

# Создание датафрейма с dummy variables
def create_df(dic, feature_list):
    out = pd.DataFrame(dic)
    out = pd.concat([out, pd.get_dummies(out[feature_list])], axis = 1)
    out.drop(feature_list, axis = 1, inplace = True)
    return out

# Некоторые значения признаков есть в тесте, но нет в трейне и наоборот
def intersect_features(train, test):
    common_feat = list( set(train.keys()) & set(test.keys()))
    return train[common_feat], test[common_feat]

In [2]:
features = ['Внешность', 'Алкоголь_в_напитке',
            'Уровень_красноречия', 'Потраченные_деньги']
target = ['Поедет']

**Обучающая выборка**

In [3]:
df_train = {}
df_train['Внешность'] = ['приятная', 'приятная', 'приятная', 'отталкивающая',
                         'отталкивающая', 'отталкивающая', 'приятная'] 
df_train['Алкоголь_в_напитке'] = ['да', 'да', 'нет', 'нет', 'да', 'да', 'да']
df_train['Уровень_красноречия'] = ['высокий', 'низкий', 'средний', 'средний', 'низкий',
                                   'высокий', 'средний']
df_train['Потраченные_деньги'] = ['много', 'мало', 'много', 'мало', 'много',
                                  'много', 'много']
df_train['Поедет'] = LabelEncoder().fit_transform(['+', '-', '+', '-', '-', '+', '+'])

df_train = create_df(df_train, features)
df_train

,Поедет,Внешность_отталкивающая,Внешность_приятная,Алкоголь_в_напитке_да,Алкоголь_в_напитке_нет,Уровень_красноречия_высокий,Уровень_красноречия_низкий,Уровень_красноречия_средний,Потраченные_деньги_мало,Потраченные_деньги_много
0,0,False,True,True,False,True,False,False,False,True
1,1,False,True,True,False,False,True,False,True,False
2,0,False,True,False,True,False,False,True,False,True
3,1,True,False,False,True,False,False,True,True,False
4,1,True,False,True,False,False,True,False,False,True
5,0,True,False,True,False,True,False,False,False,True
6,0,False,True,True,False,False,False,True,False,True


**Тестовая выборка**

In [4]:
df_test = {}
df_test['Внешность'] = ['приятная', 'приятная', 'отталкивающая'] 
df_test['Алкоголь_в_напитке'] = ['нет', 'да', 'да']
df_test['Уровень_красноречия'] = ['средний', 'высокий', 'средний']
df_test['Потраченные_деньги'] = ['много', 'мало', 'много']
df_test = create_df(df_test, features)
df_test

,Внешность_отталкивающая,Внешность_приятная,Алкоголь_в_напитке_да,Алкоголь_в_напитке_нет,Уровень_красноречия_высокий,Уровень_красноречия_средний,Потраченные_деньги_мало,Потраченные_деньги_много
0,False,True,False,True,False,True,False,True
1,False,True,True,False,True,False,True,False
2,True,False,True,False,False,True,False,True


In [5]:
# Некоторые значения признаков есть в тесте, но нет в трейне и наоборот
y = df_train['Поедет']
df_train, df_test = intersect_features(train=df_train, test=df_test)

df_train

,Потраченные_деньги_много,Уровень_красноречия_средний,Алкоголь_в_напитке_да,Уровень_красноречия_высокий,Внешность_отталкивающая,Алкоголь_в_напитке_нет,Потраченные_деньги_мало,Внешность_приятная
0,True,False,True,True,False,False,False,True
1,False,False,True,False,False,False,True,True
2,True,True,False,False,False,True,False,True
3,False,True,False,False,True,True,True,False
4,True,False,True,False,True,False,False,False
5,True,False,True,True,True,False,False,False
6,True,True,True,False,False,False,False,True


In [6]:
df_test

,Потраченные_деньги_много,Уровень_красноречия_средний,Алкоголь_в_напитке_да,Уровень_красноречия_высокий,Внешность_отталкивающая,Алкоголь_в_напитке_нет,Потраченные_деньги_мало,Внешность_приятная
0,True,True,False,False,False,True,False,True
1,False,False,True,True,False,False,True,True
2,True,True,True,False,True,False,False,False


Для облегчения рекомендуется использовать (и реализовать) функции приведённые ниже.

In [10]:
import math
from operator import countOf


# расчёт энтропии множества
def entropy(a_list):
    '''
    Calculates and returns entropy of given binary
    1d collection, using
    entropy = -p_0*log2(p_0) - p_1*log2(p_1),
    where
        p_0 - proportion of value 0 in a_list,
        p_1 - proportion of value 1 in a_list,
    '''
    count = len(a_list)
    if count == 0:
        return 0

    count_0 = countOf(a_list, 0)
    count_1 = count - count_0

    p_0 = count_0 / count
    p_1 = count_1 / count

    ent = 0
    if p_0 > 0:
        ent -= p_0 * math.log2(p_0)
    if p_1 > 0:
        ent -= p_1 * math.log2(p_1)

    return ent


In [13]:
# расчет прироста информации

def information_gain(root, left, right):
    '''
    Calculates and returns Information Gain 
    if we split root by left and right
    '''
    p_left = len(left)/len(root)
    p_right = len(right)/len(root)

    return entropy(root) - p_left * entropy(left) - p_right * entropy(right)

information_gain([0, 1, 0, 1, 1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 1, 1])

0.5487949406953987

In [27]:
# определение оптимального разбиения с точки зрения прироста информации

def best_feature_to_split(X, y, to_print=True):
    ''' Выводит прирост информации при разбиении по каждому признаку'''
    best_feature = None
    best_ig = -1

    for feature in X.columns:
        values = X[feature]
        thresholds = sorted(values.unique())

        max_ig = -1

        for t in thresholds:
            y_left = y[values <= t]
            y_right = y[values > t]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            ig = information_gain(y, y_left.tolist(), y_right.tolist())
            
            if max_ig < ig:
                max_ig = ig

        if to_print:
            print(f"splitting by {feature} gives {max_ig} information gain")

        if max_ig > best_ig:
            best_ig = max_ig
            best_feature = feature

    return best_feature


best_feature_to_split(df_train, y)

splitting by Потраченные_деньги_много gives 0.46956521111470695 information gain
splitting by Уровень_красноречия_средний gives 0.020244207153756077 information gain
splitting by Алкоголь_в_напитке_да gives 0.0059777114237740125 information gain
splitting by Уровень_красноречия_высокий gives 0.2916919971380597 information gain
splitting by Внешность_отталкивающая gives 0.12808527889139443 information gain
splitting by Алкоголь_в_напитке_нет gives 0.0059777114237740125 information gain
splitting by Потраченные_деньги_мало gives 0.46956521111470695 information gain
splitting by Внешность_приятная gives 0.12808527889139443 information gain


'Потраченные_деньги_много'

In [35]:
class DecisionTree:
    """
    Decision tree, that considers binary features and solves
    binary classification task
    """
    class Node:
        """
        Node which are used to build DecisionTree
        """
        def __init__(self, is_leaf=False, val=None, feature=None, depth=None):
            """
            Initalizes a new tree node instance
            
            Parameters
            ----------
            is_leaf, Boolean
                defines whethes it's leaf node (has no children)
            val, int
                prediction value to be stored if it's leaf node
            feature, str
                name of feature which is used to split data
                by this node
            depth, int
                depth level of this node
            """
            self.left = None
            self.right = None
            self.is_leaf = is_leaf
            self.val = val
            self.feature = feature
            self.depth = depth
            pass
                      
    def __init__(self, max_depth=3):
        """
        Initalizes a new DecisionTree instance
        """
        self.max_depth=max_depth
        self.root = None
        pass
    
    def build_tree(self, X, y, curr_node=Node(), curr_depth=0):
        """
        Returns recursively built classification tree
        """
        if curr_depth == self.max_depth or len(y.unique()) == 1:
            curr_node.is_leaf = True
            curr_node.val = y.mode()[0]
            return curr_node

        col_name = best_feature_to_split(X, y, to_print=False)

        if col_name is None:
            curr_node.is_leaf = True
            curr_node.val = y.mode()[0]
            return curr_node

        curr_node.feature = col_name
        
        X_right, y_right = X[X[col_name] == 1], y[X[col_name] == 1]
        X_left, y_left = X[~X[col_name] == 1], y[~X[col_name] == 1]

        curr_node.left = self.build_tree(X_left, y_left, self.Node(val=False, depth=curr_depth+1))
        curr_node.right = self.build_tree(X_right, y_right, self.Node(val=True, depth=curr_depth+1))
        
        return curr_node
        
    def fit(self, X, y):
        """
        Fits to X: builds a decision tree that is
        appropriate to X
        Returns self.
        """
        self.root = self.build_tree(X, y)
        return self

    def predict(self, X):
        """
        Predicts binary target feature setting off from a tree
        that is built at 'fit' step
        Returns list of predicted values.
        """
        y_pred = []
        for idx in X.index:
            node = self.root
            while not node.is_leaf:
                col_name = node.feature
                if X.loc[idx, col_name] == True:
                    node = node.right
                else:
                    node = node.left
            y_pred.append(node.val)
        return y_pred

In [36]:
tree_object = DecisionTree()

In [37]:
tree_object.fit(df_train, y)

In [38]:
tree_object.predict(df_test)

[0, 1, 1]